# Set-up Notebook

## Initialize Hail (run this on start up)

In [ ]:
import hail as hl
import pandas as pd
import numpy as np
import hail as hl
from bokeh.io import show, output_notebook
from bokeh.layouts import gridplot

hl.init(default_reference='GRCh38', idempotent=True)

## Set up files for analysis (run once only, skip on subsequent runs)

### Copy combined VCF

Add path to pipeline output combined VCF and a GCS bucket path for where to put the untarred unzipped version

In [ ]:
combined_vcf = ''
combined_vcf_mt_cloud_path = ''

Uncomment the code below and run once per analysis. Once copied, the code does not need to be run again 

In [ ]:
## 0) Make a tmp directories to hold the expanded matrix tables
#! mkdir -p ./tmp/vcf_mt/

## 1) Copy tar files to the VM
#! gsutil cp {combined_vcf} ./tmp/vcf_mt/

## 2) Unzip the tar files
#! gunzip ./tmp/vcf_mt/combined_vcf.tar.gz

## 3) Extract to a directory
#! tar -xf ./tmp/vcf_mt/combined_vcf.tar -C ./tmp/vcf_mt/

## 4) Copy the extracted MT back to GCS
#! gsutil -m cp -r ./tmp/vcf_mt/combined_vcf.mt {combined_vcf_mt_cloud_path}

## Initialize Matrices (run this)


Fill in the paths to the google storage locations in the code below. Note that the paths should take the following forms: 
* vcf_mt = 'gs://.../combined_vcf/combined_vcf.mt/')
* annotated_mt = 'gs://.../annotated_mt/annotated_combined.mt/')
* filt_annotated_mt = 'gs://.../filt_annotated_mt/annotated_combined.mt/')

In [ ]:
# New 50k sample outputs (test) - fill in the google cloud storage paths to the unzipped matrix tables
test_vcf_mt_50k = hl.read_matrix_table('')
test_annotated_mt_50k = hl.read_matrix_table('')
test_filt_annotated_mt_50k = hl.read_matrix_table('')

In [ ]:
# Previously validated 50k sample outputs (truth)
truth_vcf_mt_50k = hl.read_matrix_table('')
truth_annotated_mt_50k = hl.read_matrix_table('')
truth_filt_annotated_mt_50k = hl.read_matrix_table('')

# Compare combined VCF outputs

## High Level Comparison of VCF Matrix Tables

In [ ]:
truth_vcf_mt = truth_vcf_mt_50k
test_vcf_mt = test_vcf_mt_50k

In [ ]:
# Split multi-allelic sites
truth_vcf_mt = hl.split_multi_hts(truth_vcf_mt)
test_vcf_mt = hl.split_multi_hts(test_vcf_mt)

## Comparison code

In [ ]:
import hail as hl

def compare_matrix_tables(mt1, mt2, output_prefix="mt_comparison", check_entries=True):
    """
    Compare 2 Hail matrix tables expected to be identical.
    
    Parameters:
    -----------
    mt1, mt2 : MatrixTable
        The two matrix tables to compare
    output_prefix : str
        Prefix for output files
    check_entries : bool
        Whether to perform detailed entry-level comparison (can be slow)
    
    Returns:
    --------
    dict : Comparison results with boolean flags for each check
    """
    
    results = {}
    
    print("=" * 80)
    print("MATRIX TABLE COMPARISON")
    print("=" * 80)
    
    # 1. Compare dimensions
    print("\n1. DIMENSIONS:")
    print("-" * 80)
    n_rows_1 = mt1.count_rows()
    n_cols_1 = mt1.count_cols()
    n_rows_2 = mt2.count_rows()
    n_cols_2 = mt2.count_cols()
    
    print(f"MT1: {n_rows_1:,} rows × {n_cols_1:,} columns")
    print(f"MT2: {n_rows_2:,} rows × {n_cols_2:,} columns")
    
    dims_match = (n_rows_1 == n_rows_2) and (n_cols_1 == n_cols_2)
    results['dimensions_match'] = dims_match
    
    if dims_match:
        print("✓ Dimensions match")
    else:
        print("✗ Dimensions differ")
        if n_rows_1 != n_rows_2:
            print(f"  Row difference: {abs(n_rows_1 - n_rows_2):,}")
        if n_cols_1 != n_cols_2:
            print(f"  Column difference: {abs(n_cols_1 - n_cols_2):,}")
    
    # 2. Compare schemas
    print("\n2. SCHEMA COMPARISON:")
    print("-" * 80)
    schema1 = mt1.describe(handler=lambda x: x)
    schema2 = mt2.describe(handler=lambda x: x)
    
    schemas_match = (schema1 == schema2)
    results['schemas_match'] = schemas_match
    
    if schemas_match:
        print("✓ Schemas are identical")
    else:
        print("✗ Schemas differ")
        print("\nMT1 schema:")
        mt1.describe()
        print("\nMT2 schema:")
        mt2.describe()
    
    # 3. Compare column keys (samples)
    print("\n3. COLUMN KEYS (SAMPLES):")
    print("-" * 80)
    samples1 = set(mt1.s.collect())
    samples2 = set(mt2.s.collect())
    
    print(f"MT1: {len(samples1)} samples")
    print(f"MT2: {len(samples2)} samples")
    
    samples_match = (samples1 == samples2)
    results['samples_match'] = samples_match
    
    if samples_match:
        print("✓ Sample sets are identical")
    else:
        print("✗ Sample sets differ")
        only_in_mt1 = samples1 - samples2
        only_in_mt2 = samples2 - samples1
        
        if only_in_mt1:
            print(f"\n  Samples only in MT1 ({len(only_in_mt1)}):")
            for s in sorted(list(only_in_mt1)[:10]):
                print(f"    {s}")
            if len(only_in_mt1) > 10:
                print(f"    ... and {len(only_in_mt1) - 10} more")
        
        if only_in_mt2:
            print(f"\n  Samples only in MT2 ({len(only_in_mt2)}):")
            for s in sorted(list(only_in_mt2)[:10]):
                print(f"    {s}")
            if len(only_in_mt2) > 10:
                print(f"    ... and {len(only_in_mt2) - 10} more")
    
    # 4. Compare row keys (variants)
    print("\n4. ROW KEYS (VARIANTS):")
    print("-" * 80)
    print("Collecting row keys (this may take a moment)...")
    
    # Get row keys
    rows1 = mt1.rows()
    rows2 = mt2.rows()
    
    # Create comparable row key sets
    row_keys1 = rows1.select().collect()
    row_keys2 = rows2.select().collect()

    row_key_set1 = set((r.locus, tuple(r.alleles)) for r in row_keys1)
    row_key_set2 = set((r.locus, tuple(r.alleles)) for r in row_keys2)
    
    print(f"MT1: {len(row_key_set1):,} unique variants")
    print(f"MT2: {len(row_key_set2):,} unique variants")
    
    variants_match = (row_key_set1 == row_key_set2)
    results['variants_match'] = variants_match
    
    if variants_match:
        print("✓ Variant sets are identical")
    else:
        print("✗ Variant sets differ")
        only_in_mt1 = row_key_set1 - row_key_set2
        only_in_mt2 = row_key_set2 - row_key_set1
        
        if only_in_mt1:
            print(f"\n  Variants only in MT1: {len(only_in_mt1):,}")
            print(f"  First few examples:")
            for variant in sorted(list(only_in_mt1)[:5]):
                print(f"    {variant[0]}, {variant[1]}")
        
        if only_in_mt2:
            print(f"\n  Variants only in MT2: {len(only_in_mt2):,}")
            print(f"  First few examples:")
            for variant in sorted(list(only_in_mt2)[:5]):
                print(f"    {variant[0]}, {variant[1]}")
        
        # Export variant differences
        if only_in_mt1:
            diff_file = f"{output_prefix}_variants_only_in_MT1.tsv"
            rows1.filter(hl.literal(only_in_mt1).contains(
                hl.tuple([rows1.locus, hl.tuple(rows1.alleles)])
            )).export(diff_file)
            print(f"\n  Variants only in MT1 exported to: {diff_file}")
        
        if only_in_mt2:
            diff_file = f"{output_prefix}_variants_only_in_MT2.tsv"
            rows2.filter(hl.literal(only_in_mt2).contains(
                hl.tuple([rows2.locus, hl.tuple(rows2.alleles)])
            )).export(diff_file)
            print(f"  Variants only in MT2 exported to: {diff_file}")
    
    # 5. Compare row fields (for shared variants)
    if variants_match:
        print("\n5. ROW FIELD COMPARISON:")
        print("-" * 80)
        print("Comparing row fields (filters, a_index, was_split)...")
        
        # Join row tables on key
        rt_joined = rows1.annotate(
            mt2_filters=rows2[rows1.key].filters,
            mt2_a_index=rows2[rows1.key].a_index,
            mt2_was_split=rows2[rows1.key].was_split,
        )
        
        # Find differences
        rt_diff = rt_joined.filter(
            (rt_joined.filters != rt_joined.mt2_filters) |
            (rt_joined.a_index != rt_joined.mt2_a_index) |
            (rt_joined.was_split != rt_joined.mt2_was_split)
        )
        
        n_row_diff = rt_diff.count()
        row_fields_match = (n_row_diff == 0)
        results['row_fields_match'] = row_fields_match
        
        if row_fields_match:
            print("✓ All row fields match")
        else:
            print(f"✗ {n_row_diff:,} variants have differing row fields")
            
            # Export differences
            diff_file = f"{output_prefix}_row_field_differences.tsv"
            rt_diff.export(diff_file)
            print(f"  Differences exported to: {diff_file}")
            
            # Show summary of differences
            diff_summary = rt_diff.aggregate(hl.struct(
                filters_diff=hl.agg.count_where(rt_diff.filters != rt_diff.mt2_filters),
                a_index_diff=hl.agg.count_where(rt_diff.a_index != rt_diff.mt2_a_index),
                was_split_diff=hl.agg.count_where(rt_diff.was_split != rt_diff.mt2_was_split)
            ))
            
            print(f"\n  Field-specific differences:")
            print(f"    filters: {diff_summary.filters_diff:,} variants")
            print(f"    a_index: {diff_summary.a_index_diff:,} variants")
            print(f"    was_split: {diff_summary.was_split_diff:,} variants")
    
    # 6. Compare entry fields (if both dimensions and keys match)
    if check_entries and samples_match and variants_match:
        print("\n6. ENTRY FIELD COMPARISON:")
        print("-" * 80)
        print("Comparing entry values (this may take a while)...")
        
        # Annotate mt1 with mt2 entry fields
        mt_joined = mt1.annotate_entries(
            mt2_DP=mt2[mt1.row_key, mt1.col_key].DP,
            mt2_AD=mt2[mt1.row_key, mt1.col_key].AD,
            mt2_OriginalSelfRefAlleles=mt2[mt1.row_key, mt1.col_key].OriginalSelfRefAlleles,
            mt2_SwappedFieldIDs=mt2[mt1.row_key, mt1.col_key].SwappedFieldIDs,
            mt2_F2R1=mt2[mt1.row_key, mt1.col_key].F2R1,
            mt2_F1R2=mt2[mt1.row_key, mt1.col_key].F1R2,
            mt2_MPOS=mt2[mt1.row_key, mt1.col_key].MPOS,
            mt2_AS_SB_TABLE=mt2[mt1.row_key, mt1.col_key].AS_SB_TABLE,
            mt2_STR=mt2[mt1.row_key, mt1.col_key].STR,
            mt2_STRQ=mt2[mt1.row_key, mt1.col_key].STRQ,
            mt2_RPA=mt2[mt1.row_key, mt1.col_key].RPA,
            mt2_HL=mt2[mt1.row_key, mt1.col_key].HL,
            mt2_MQ=mt2[mt1.row_key, mt1.col_key].MQ,
            mt2_TLOD=mt2[mt1.row_key, mt1.col_key].TLOD,
            mt2_FT=mt2[mt1.row_key, mt1.col_key].FT,
        )
        
        # Filter to entries with differences
        mt_diff = mt_joined.filter_entries(
            (mt_joined.DP != mt_joined.mt2_DP) |
            (mt_joined.AD != mt_joined.mt2_AD) |
            (mt_joined.OriginalSelfRefAlleles != mt_joined.mt2_OriginalSelfRefAlleles) |
            (mt_joined.SwappedFieldIDs != mt_joined.mt2_SwappedFieldIDs) |
            (mt_joined.F2R1 != mt_joined.mt2_F2R1) |
            (mt_joined.F1R2 != mt_joined.mt2_F1R2) |
            (mt_joined.MPOS != mt_joined.mt2_MPOS) |
            (mt_joined.AS_SB_TABLE != mt_joined.mt2_AS_SB_TABLE) |
            (mt_joined.STR != mt_joined.mt2_STR) |
            (mt_joined.STRQ != mt_joined.mt2_STRQ) |
            (mt_joined.RPA != mt_joined.mt2_RPA) |
            (mt_joined.HL != mt_joined.mt2_HL) |
            (mt_joined.MQ != mt_joined.mt2_MQ) |
            (mt_joined.TLOD != mt_joined.mt2_TLOD) |
            (mt_joined.FT != mt_joined.mt2_FT)
        )
        
        # Count differences
        n_diff_entries = mt_diff.entries().count()
        entries_match = (n_diff_entries == 0)
        results['entries_match'] = entries_match
        
        if entries_match:
            print("✓ All entry fields match")
        else:
            print(f"✗ {n_diff_entries:,} entries have differing values")
            
            # Export differences
            diff_file = f"{output_prefix}_entry_differences.tsv"
            mt_diff.entries().export(diff_file)
            print(f"  Differences exported to: {diff_file}")
            
            # Get per-field difference counts
            print("\n  Computing per-field difference counts...")
            field_diffs = mt_joined.aggregate_entries(hl.struct(
                DP=hl.agg.count_where(mt_joined.DP != mt_joined.mt2_DP),
                AD=hl.agg.count_where(mt_joined.AD != mt_joined.mt2_AD),
                F2R1=hl.agg.count_where(mt_joined.F2R1 != mt_joined.mt2_F2R1),
                F1R2=hl.agg.count_where(mt_joined.F1R2 != mt_joined.mt2_F1R2),
                MPOS=hl.agg.count_where(mt_joined.MPOS != mt_joined.mt2_MPOS),
                TLOD=hl.agg.count_where(mt_joined.TLOD != mt_joined.mt2_TLOD),
                FT=hl.agg.count_where(mt_joined.FT != mt_joined.mt2_FT),
                MQ=hl.agg.count_where(mt_joined.MQ != mt_joined.mt2_MQ),
                HL=hl.agg.count_where(mt_joined.HL != mt_joined.mt2_HL),
            ))
            
            print(f"\n  Field-specific differences:")
            for field, count in field_diffs.items():
                if count > 0:
                    print(f"    {field}: {count:,} entries")
    
    elif check_entries and (not samples_match or not variants_match):
        print("\n6. ENTRY FIELD COMPARISON:")
        print("-" * 80)
        print("⊘ Skipping entry comparison (samples or variants don't match)")
        results['entries_match'] = None
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY:")
    print("=" * 80)
    
    all_match = all(v == True for v in results.values() if v is not None)
    
    if all_match:
        print("✓ Matrix tables are IDENTICAL")
    else:
        print("✗ Matrix tables DIFFER")
        print("\nDifferences found in:")
        for check, matches in results.items():
            if matches == False:
                print(f"  - {check}")
    
    return results

    
# Run comparison
results = compare_matrix_tables(
    truth_vcf_mt, 
    test_vcf_mt, 
    output_prefix="my_comparison",
    check_entries=True  # Set to False to skip entry comparison
)
    
# Access results
print(f"\nAre they identical? {all(v == True for v in results.values() if v is not None)}")

# Compare Annotated Output (unfiltered)

In [ ]:
truth_annotated_mt = truth_annotated_mt_50k
test_annotated_mt = test_annotated_mt_50k

#### Compare Schemas

In [ ]:
def compare_schemas(mt1, mt2, mt1_name="MT1", mt2_name="MT2"):
    """
    Systematically compare schemas of two Hail matrix tables.
    
    Parameters:
    -----------
    mt1, mt2 : MatrixTable
        The two matrix tables to compare
    mt1_name, mt2_name : str
        Names for the matrix tables in output
    
    Returns:
    --------
    dict : Comparison results
    """
    
    results = {
        'global_fields_match': True,
        'column_fields_match': True,
        'row_fields_match': True,
        'entry_fields_match': True,
        'keys_match': True,
        'differences': []
    }
    
    print("=" * 80)
    print(f"SCHEMA COMPARISON: {mt1_name} vs {mt2_name}")
    print("=" * 80)
    
    # Helper function to get field types as dict
    def get_field_dict(fields):
        return {name: str(dtype) for name, dtype in fields.items()}
    
    # 1. Compare Global Fields
    print("\n1. GLOBAL FIELDS:")
    print("-" * 80)
    
    global1 = get_field_dict(mt1.globals.dtype)
    global2 = get_field_dict(mt2.globals.dtype)
    
    if global1 == global2:
        print(f"✓ Global fields match ({len(global1)} fields)")
    else:
        print(f"✗ Global fields differ")
        results['global_fields_match'] = False
        
        only_in_mt1 = set(global1.keys()) - set(global2.keys())
        only_in_mt2 = set(global2.keys()) - set(global1.keys())
        common = set(global1.keys()) & set(global2.keys())
        type_diffs = [f for f in common if global1[f] != global2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {global1[field]}")
                results['differences'].append(f"Global field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {global2[field]}")
                results['differences'].append(f"Global field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {global1[field]}")
                print(f"      {mt2_name}: {global2[field]}")
                results['differences'].append(f"Global field '{field}' has different types")
    
    # 2. Compare Column Fields
    print("\n2. COLUMN FIELDS:")
    print("-" * 80)
    
    col1 = get_field_dict(mt1.col.dtype)
    col2 = get_field_dict(mt2.col.dtype)
    
    if col1 == col2:
        print(f"✓ Column fields match ({len(col1)} fields)")
    else:
        print(f"✗ Column fields differ")
        results['column_fields_match'] = False
        
        only_in_mt1 = set(col1.keys()) - set(col2.keys())
        only_in_mt2 = set(col2.keys()) - set(col1.keys())
        common = set(col1.keys()) & set(col2.keys())
        type_diffs = [f for f in common if col1[f] != col2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {col1[field]}")
                results['differences'].append(f"Column field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {col2[field]}")
                results['differences'].append(f"Column field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {col1[field]}")
                print(f"      {mt2_name}: {col2[field]}")
                results['differences'].append(f"Column field '{field}' has different types")
    
    # 3. Compare Row Fields
    print("\n3. ROW FIELDS:")
    print("-" * 80)
    
    row1 = get_field_dict(mt1.row.dtype)
    row2 = get_field_dict(mt2.row.dtype)
    
    if row1 == row2:
        print(f"✓ Row fields match ({len(row1)} fields)")
    else:
        print(f"✗ Row fields differ")
        results['row_fields_match'] = False
        
        only_in_mt1 = set(row1.keys()) - set(row2.keys())
        only_in_mt2 = set(row2.keys()) - set(row1.keys())
        common = set(row1.keys()) & set(row2.keys())
        type_diffs = [f for f in common if row1[f] != row2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {row1[field]}")
                results['differences'].append(f"Row field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {row2[field]}")
                results['differences'].append(f"Row field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {row1[field]}")
                print(f"      {mt2_name}: {row2[field]}")
                results['differences'].append(f"Row field '{field}' has different types")
    
    # 4. Compare Entry Fields
    print("\n4. ENTRY FIELDS:")
    print("-" * 80)
    
    entry1 = get_field_dict(mt1.entry.dtype)
    entry2 = get_field_dict(mt2.entry.dtype)
    
    if entry1 == entry2:
        print(f"✓ Entry fields match ({len(entry1)} fields)")
    else:
        print(f"✗ Entry fields differ")
        results['entry_fields_match'] = False
        
        only_in_mt1 = set(entry1.keys()) - set(entry2.keys())
        only_in_mt2 = set(entry2.keys()) - set(entry1.keys())
        common = set(entry1.keys()) & set(entry2.keys())
        type_diffs = [f for f in common if entry1[f] != entry2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {entry1[field]}")
                results['differences'].append(f"Entry field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {entry2[field]}")
                results['differences'].append(f"Entry field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {entry1[field]}")
                print(f"      {mt2_name}: {entry2[field]}")
                results['differences'].append(f"Entry field '{field}' has different types")
    
    # 5. Compare Keys
    print("\n5. KEY FIELDS:")
    print("-" * 80)
    
    col_key1 = list(mt1.col_key.dtype)
    col_key2 = list(mt2.col_key.dtype)
    row_key1 = list(mt1.row_key.dtype)
    row_key2 = list(mt2.row_key.dtype)
    
    keys_match = True
    
    if col_key1 == col_key2:
        print(f"✓ Column keys match: {list(mt1.col_key.dtype.keys())}")
    else:
        print(f"✗ Column keys differ")
        print(f"  {mt1_name}: {list(mt1.col_key.dtype.keys())}")
        print(f"  {mt2_name}: {list(mt2.col_key.dtype.keys())}")
        results['differences'].append("Column keys differ")
        keys_match = False
    
    if row_key1 == row_key2:
        print(f"✓ Row keys match: {list(mt1.row_key.dtype.keys())}")
    else:
        print(f"✗ Row keys differ")
        print(f"  {mt1_name}: {list(mt1.row_key.dtype.keys())}")
        print(f"  {mt2_name}: {list(mt2.row_key.dtype.keys())}")
        results['differences'].append("Row keys differ")
        keys_match = False
    
    results['keys_match'] = keys_match
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY:")
    print("=" * 80)
    
    all_match = (results['global_fields_match'] and 
                 results['column_fields_match'] and 
                 results['row_fields_match'] and 
                 results['entry_fields_match'] and 
                 results['keys_match'])
    
    if all_match:
        print("✓ Schemas are IDENTICAL")
    else:
        print("✗ Schemas DIFFER")
        print(f"\nTotal differences found: {len(results['differences'])}")
        
        if not results['global_fields_match']:
            print("  - Global fields differ")
        if not results['column_fields_match']:
            print("  - Column fields differ")
        if not results['row_fields_match']:
            print("  - Row fields differ")
        if not results['entry_fields_match']:
            print("  - Entry fields differ")
        if not results['keys_match']:
            print("  - Keys differ")
    
    return results


# Example usage:
if __name__ == "__main__":
    
    # Compare schemas
    schema_results = compare_schemas(truth_annotated_mt, test_annotated_mt, "Original MT", "New MT")
    
    # Access results
    print(f"\nSchemas identical? {all(schema_results[k] for k in ['global_fields_match', 'column_fields_match', 'row_fields_match', 'entry_fields_match', 'keys_match'])}")

#### Thourough comparison

In [ ]:
import hail as hl
import os

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def get_output_path(filename):
    """Get full path in current working directory."""
    cwd = os.getcwd()
    full_path = os.path.join(cwd, filename)
    print(f"  File will be written to: {full_path}")
    return full_path


def write_table_to_file(table, filename):
    """Write a Hail table to TSV file in current directory."""
    output_path = get_output_path(filename)
    
    # Count rows
    n_rows = table.count()
    print(f"  Writing {n_rows} rows...")
    
    # WORKAROUND: Collect to Python and write manually
    # This works when .export() fails
    print(f"  Collecting data to Python...")
    
    # Get field names
    fields = list(table.row.dtype.keys())
    
    # Collect the data
    data = table.collect()
    
    print(f"  Writing to file...")
    with open(output_path, 'w') as f:
        # Write header
        f.write('\t'.join(fields) + '\n')
        
        # Write data rows
        for row in data:
            values = []
            for field in fields:
                val = row[field]
                # Convert to string, handling None/missing values
                if val is None:
                    values.append('NA')
                else:
                    values.append(str(val))
            f.write('\t'.join(values) + '\n')
    
    # Verify file was created
    if os.path.exists(output_path):
        size = os.path.getsize(output_path)
        print(f"  ✓ File created successfully ({size:,} bytes)")
        return output_path
    else:
        print(f"  ✗ WARNING: File still not found after manual write!")
        return None
    
    
# =============================================================================
# COMPARISON FUNCTIONS
# =============================================================================

def compare_dimensions(mt1, mt2, mt1_name="MT1", mt2_name="MT2"):
    """Compare matrix table dimensions."""
    print("\n" + "=" * 80)
    print("1. DIMENSIONS")
    print("=" * 80)
    
    n_rows_1 = mt1.count_rows()
    n_cols_1 = mt1.count_cols()
    n_rows_2 = mt2.count_rows()
    n_cols_2 = mt2.count_cols()
    
    print(f"{mt1_name}: {n_rows_1:,} rows × {n_cols_1:,} columns")
    print(f"{mt2_name}: {n_rows_2:,} rows × {n_cols_2:,} columns")
    
    dims_match = (n_rows_1 == n_rows_2) and (n_cols_1 == n_cols_2)
    
    if dims_match:
        print("✓ Dimensions match")
    else:
        print("✗ Dimensions differ")
        if n_rows_1 != n_rows_2:
            print(f"  Row difference: {abs(n_rows_1 - n_rows_2):,}")
        if n_cols_1 != n_cols_2:
            print(f"  Column difference: {abs(n_cols_1 - n_cols_2):,}")
    
    return {
        'match': dims_match,
        'n_rows_1': n_rows_1,
        'n_cols_1': n_cols_1,
        'n_rows_2': n_rows_2,
        'n_cols_2': n_cols_2
    }


def compare_global_fields(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare global fields."""
    print("\n" + "=" * 80)
    print("2. GLOBAL FIELDS")
    print("=" * 80)
    
    globals1 = mt1.globals.collect()[0]
    globals2 = mt2.globals.collect()[0]
    
    global_fields = list(mt1.globals.dtype.keys())
    print(f"Comparing {len(global_fields)} global fields...")
    
    global_diffs = []
    diff_details = []
    
    for field in global_fields:
        val1 = globals1[field]
        val2 = globals2[field]
        if val1 != val2:
            global_diffs.append(field)
            diff_details.append({
                'field': field,
                f'{mt1_name}_value': str(val1),
                f'{mt2_name}_value': str(val2)
            })
    
    globals_match = len(global_diffs) == 0
    
    if globals_match:
        print("✓ All global fields match")
    else:
        print(f"✗ {len(global_diffs)} global fields differ:")
        for field in global_diffs:
            print(f"  - {field}")
        
        # Write differences to file
        filename = f"{output_prefix}_global_field_differences.txt"
        output_path = get_output_path(filename)
        with open(output_path, 'w') as f:
            f.write(f"Global Field Differences: {mt1_name} vs {mt2_name}\n")
            f.write("=" * 80 + "\n\n")
            for detail in diff_details:
                f.write(f"Field: {detail['field']}\n")
                f.write(f"  {mt1_name}: {detail[f'{mt1_name}_value']}\n")
                f.write(f"  {mt2_name}: {detail[f'{mt2_name}_value']}\n")
                f.write("\n")
        
        if os.path.exists(output_path):
            print(f"  ✓ Differences written to: {output_path}")
    
    return {
        'match': globals_match,
        'n_diffs': len(global_diffs),
        'diff_fields': global_diffs
    }


def compare_samples(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare column keys (samples)."""
    print("\n" + "=" * 80)
    print("3. COLUMN KEYS (SAMPLES)")
    print("=" * 80)
    
    samples1 = mt1.s.collect()
    samples2 = mt2.s.collect()
    
    samples1_set = set(samples1)
    samples2_set = set(samples2)
    
    print(f"{mt1_name}: {len(samples1)} samples")
    print(f"{mt2_name}: {len(samples2)} samples")
    
    samples_match = (samples1_set == samples2_set)
    sample_order_match = (samples1 == samples2)
    
    if samples_match:
        print("✓ Sample sets are identical")
        if sample_order_match:
            print("✓ Sample order is identical")
        else:
            print("⚠ Sample order differs (but same samples present)")
    else:
        print("✗ Sample sets differ")
        
        only_in_mt1 = samples1_set - samples2_set
        only_in_mt2 = samples2_set - samples1_set
        
        if only_in_mt1:
            print(f"\n  Samples only in {mt1_name}: {len(only_in_mt1)}")
            filename = f"{output_prefix}_samples_only_in_{mt1_name}.txt"
            output_path = get_output_path(filename)
            with open(output_path, 'w') as f:
                for s in sorted(only_in_mt1):
                    f.write(f"{s}\n")
            if os.path.exists(output_path):
                print(f"  ✓ List written to: {output_path}")
        
        if only_in_mt2:
            print(f"\n  Samples only in {mt2_name}: {len(only_in_mt2)}")
            filename = f"{output_prefix}_samples_only_in_{mt2_name}.txt"
            output_path = get_output_path(filename)
            with open(output_path, 'w') as f:
                for s in sorted(only_in_mt2):
                    f.write(f"{s}\n")
            if os.path.exists(output_path):
                print(f"  ✓ List written to: {output_path}")
    
    return {
        'match': samples_match,
        'order_match': sample_order_match,
        'n_samples_1': len(samples1),
        'n_samples_2': len(samples2),
        'n_only_in_1': len(samples1_set - samples2_set),
        'n_only_in_2': len(samples2_set - samples1_set)
    }


def compare_column_fields(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare column fields (sample metadata)."""
    print("\n" + "=" * 80)
    print("4. COLUMN FIELDS (SAMPLE METADATA)")
    print("=" * 80)
    
    # First check if samples match
    samples1 = set(mt1.s.collect())
    samples2 = set(mt2.s.collect())
    
    if samples1 != samples2:
        print("⊘ Skipping column field comparison (samples don't match)")
        return {'match': None, 'skipped': True}
    
    cols1 = mt1.cols()
    cols2 = mt2.cols()
    
    # Get all column fields except the key
    col_fields = [f for f in mt1.col.dtype.keys() if f not in mt1.col_key.dtype.keys()]
    
    print(f"Comparing {len(col_fields)} column fields across {len(samples1)} samples...")
    
    # Join the column tables
    cols_joined = cols1.annotate(**{f"mt2_{field}": cols2[cols1.key][field] 
                                    for field in col_fields})
    
    # Check for differences in each field
    col_field_diffs = {}
    for field in col_fields:
        diff_count = cols_joined.aggregate(
            hl.agg.count_where(cols_joined[field] != cols_joined[f"mt2_{field}"])
        )
        if diff_count > 0:
            col_field_diffs[field] = diff_count
    
    col_fields_match = len(col_field_diffs) == 0
    
    if col_fields_match:
        print("✓ All column fields match across all samples")
    else:
        print(f"✗ {len(col_field_diffs)} column fields have differences:")
        for field, count in col_field_diffs.items():
            print(f"  - {field}: {count:,} samples differ")
        
        # Export differences
        cols_diff = cols_joined.filter(
            hl.any([cols_joined[field] != cols_joined[f"mt2_{field}"] 
                   for field in col_field_diffs.keys()])
        )
        
        filename = f"{output_prefix}_column_field_differences.tsv"
        write_table_to_file(cols_diff, filename)
    
    return {
        'match': col_fields_match,
        'n_fields_checked': len(col_fields),
        'n_fields_diff': len(col_field_diffs),
        'diff_fields': col_field_diffs
    }


def compare_variants(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare row keys (variants)."""
    print("\n" + "=" * 80)
    print("5. ROW KEYS (VARIANTS)")
    print("=" * 80)
    print("Collecting row keys...")
    
    # Get dimensions
    n_rows_1 = mt1.count_rows()
    n_rows_2 = mt2.count_rows()
    
    print(f"{mt1_name}: {n_rows_1:,} variants")
    print(f"{mt2_name}: {n_rows_2:,} variants")
    
    # Use rows() and select just the keys
    rows1 = mt1.rows().select()
    rows2 = mt2.rows().select()
    
    # Use anti-join to find variants only in each
    only_in_mt1 = rows1.anti_join(rows2)
    only_in_mt2 = rows2.anti_join(rows1)
    
    n_only_mt1 = only_in_mt1.count()
    n_only_mt2 = only_in_mt2.count()
    
    variants_match = (n_only_mt1 == 0) and (n_only_mt2 == 0)
    
    if variants_match:
        print("✓ Variant sets are identical")
    else:
        print("✗ Variant sets differ")
        
        if n_only_mt1 > 0:
            print(f"\n  Variants only in {mt1_name}: {n_only_mt1:,}")
            filename = f"{output_prefix}_variants_only_in_{mt1_name}.tsv"
            write_table_to_file(only_in_mt1, filename)
        
        if n_only_mt2 > 0:
            print(f"\n  Variants only in {mt2_name}: {n_only_mt2:,}")
            filename = f"{output_prefix}_variants_only_in_{mt2_name}.tsv"
            write_table_to_file(only_in_mt2, filename)
    
    return {
        'match': variants_match,
        'n_variants_1': n_rows_1,
        'n_variants_2': n_rows_2,
        'n_only_in_1': n_only_mt1,
        'n_only_in_2': n_only_mt2
    }


def compare_row_fields(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare row fields (variant annotations)."""
    print("\n" + "=" * 80)
    print("6. ROW FIELDS (VARIANT ANNOTATIONS)")
    print("=" * 80)
    
    # First check if variants match
    rows1 = mt1.rows().select()
    rows2 = mt2.rows().select()
    only_in_mt1 = rows1.anti_join(rows2)
    only_in_mt2 = rows2.anti_join(rows1)
    
    if only_in_mt1.count() > 0 or only_in_mt2.count() > 0:
        print("⊘ Skipping row field comparison (variants don't match)")
        return {'match': None, 'skipped': True}
    
    rows1_full = mt1.rows()
    rows2_full = mt2.rows()
    
    # Get all row fields except the keys
    row_fields = [f for f in mt1.row.dtype.keys() if f not in mt1.row_key.dtype.keys()]
    
    n_variants = mt1.count_rows()
    print(f"Comparing {len(row_fields)} row fields across {n_variants:,} variants...")
    
    # Join the row tables
    rows_joined = rows1_full.annotate(**{f"mt2_{field}": rows2_full[rows1_full.key][field] 
                                         for field in row_fields})
    
    # Check for differences in each field
    print("  Checking each field for differences...")
    row_field_diffs = {}
    for i, field in enumerate(row_fields, 1):
        print(f"    [{i}/{len(row_fields)}] Checking {field}...", end='\r')
        diff_count = rows_joined.aggregate(
            hl.agg.count_where(rows_joined[field] != rows_joined[f"mt2_{field}"])
        )
        if diff_count > 0:
            row_field_diffs[field] = diff_count
    
    print(" " * 80, end='\r')  # Clear the progress line
    
    row_fields_match = len(row_field_diffs) == 0
    
    if row_fields_match:
        print("✓ All row fields match across all variants")
    else:
        print(f"✗ {len(row_field_diffs)} row fields have differences:")
        for field, count in sorted(row_field_diffs.items(), key=lambda x: x[1], reverse=True):
            print(f"  - {field}: {count:,} variants differ")
        
        # Export differences
        rows_diff = rows_joined.filter(
            hl.any([rows_joined[field] != rows_joined[f"mt2_{field}"] 
                   for field in row_field_diffs.keys()])
        )
        
        filename = f"{output_prefix}_row_field_differences.tsv"
        write_table_to_file(rows_diff, filename)
    
    return {
        'match': row_fields_match,
        'n_fields_checked': len(row_fields),
        'n_fields_diff': len(row_field_diffs),
        'diff_fields': row_field_diffs
    }


def compare_entry_fields(mt1, mt2, mt1_name="MT1", mt2_name="MT2", output_prefix="comparison"):
    """Compare entry fields (genotype data)."""
    print("\n" + "=" * 80)
    print("7. ENTRY FIELDS (GENOTYPE DATA)")
    print("=" * 80)
    
    # Check if samples and variants match
    samples1 = set(mt1.s.collect())
    samples2 = set(mt2.s.collect())
    
    rows1 = mt1.rows().select()
    rows2 = mt2.rows().select()
    only_in_mt1 = rows1.anti_join(rows2)
    only_in_mt2 = rows2.anti_join(rows1)
    variants_match = (only_in_mt1.count() == 0 and only_in_mt2.count() == 0)
    
    if samples1 != samples2 or not variants_match:
        if samples1 != samples2:
            print("⊘ Skipping entry comparison (samples don't match)")
        else:
            print("⊘ Skipping entry comparison (variants don't match)")
        return {'match': None, 'skipped': True}
    
    entry_fields = list(mt1.entry.dtype.keys())
    n_rows = mt1.count_rows()
    n_cols = mt1.count_cols()
    
    print(f"Comparing {len(entry_fields)} entry fields...")
    print(f"This will check {n_rows:,} variants × {n_cols:,} samples = {n_rows * n_cols:,} entries")
    
    # Annotate mt1 with mt2 entry fields
    print("\n  Annotating entries from both matrix tables...")
    mt_joined = mt1.annotate_entries(
        **{f"mt2_{field}": mt2[mt1.row_key, mt1.col_key][field] 
           for field in entry_fields}
    )
    
    # Check for differences in each entry field
    print("\n  Checking each entry field for differences...")
    entry_field_diffs = {}
    
    for i, field in enumerate(entry_fields, 1):
        print(f"    [{i}/{len(entry_fields)}] Checking {field}...", end='\r')
        diff_count = mt_joined.aggregate_entries(
            hl.agg.count_where(mt_joined[field] != mt_joined[f"mt2_{field}"])
        )
        if diff_count > 0:
            entry_field_diffs[field] = diff_count
    
    print(" " * 80, end='\r')  # Clear the progress line
    
    entry_fields_match = len(entry_field_diffs) == 0
    
    if entry_fields_match:
        print("✓ All entry fields match across all entries")
    else:
        total_diffs = sum(entry_field_diffs.values())
        print(f"✗ {len(entry_field_diffs)} entry fields have differences ({total_diffs:,} total differing entries):")
        for field, count in sorted(entry_field_diffs.items(), key=lambda x: x[1], reverse=True):
            pct = 100 * count / (n_rows * n_cols)
            print(f"  - {field}: {count:,} entries differ ({pct:.4f}%)")
        
        # Filter to entries with differences
        print(f"\n  Filtering to entries with differences...")
        mt_diff = mt_joined.filter_entries(
            hl.any([mt_joined[field] != mt_joined[f"mt2_{field}"] 
                   for field in entry_field_diffs.keys()])
        )
        
        # Export differences - convert to table first
        filename = f"{output_prefix}_entry_differences.tsv"
        write_table_to_file(mt_diff.entries(), filename)
    
    return {
        'match': entry_fields_match,
        'n_fields_checked': len(entry_fields),
        'n_fields_diff': len(entry_field_diffs),
        'diff_fields': entry_field_diffs
    }


# =============================================================================
# MAIN COMPARISON FUNCTION
# =============================================================================

def compare_matrix_tables(mt1, mt2, mt1_name="MT1", mt2_name="MT2", 
                         output_prefix="comparison",
                         check_dimensions=True,
                         check_globals=True,
                         check_samples=True,
                         check_column_fields=True,
                         check_variants=True,
                         check_row_fields=True,
                         check_entry_fields=True):
    """
    Compare two Hail matrix tables with individual checks.
    
    Parameters:
    -----------
    mt1, mt2 : MatrixTable
        The two matrix tables to compare
    mt1_name, mt2_name : str
        Names for the matrix tables in output
    output_prefix : str
        Prefix for output files (will be written to current working directory)
    check_* : bool
        Flags to enable/disable individual checks
    
    Returns:
    --------
    dict : Results from all comparisons
    """
    
    print("=" * 80)
    print(f"MATRIX TABLE COMPARISON: {mt1_name} vs {mt2_name}")
    print("=" * 80)
    print(f"Current working directory: {os.getcwd()}")
    print(f"Output prefix: {output_prefix}")
    
    results = {}
    
    if check_dimensions:
        results['dimensions'] = compare_dimensions(mt1, mt2, mt1_name, mt2_name)
    
    if check_globals:
        results['globals'] = compare_global_fields(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    if check_samples:
        results['samples'] = compare_samples(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    if check_column_fields:
        results['column_fields'] = compare_column_fields(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    if check_variants:
        results['variants'] = compare_variants(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    if check_row_fields:
        results['row_fields'] = compare_row_fields(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    if check_entry_fields:
        results['entry_fields'] = compare_entry_fields(mt1, mt2, mt1_name, mt2_name, output_prefix)
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    
    for check_name, result in results.items():
        if result.get('skipped'):
            print(f"⊘ {check_name}: skipped")
        elif result.get('match') == True:
            print(f"✓ {check_name}: match")
        elif result.get('match') == False:
            print(f"✗ {check_name}: DIFFER")
        elif result.get('match') is None:
            print(f"⊘ {check_name}: skipped")
    
    # Check if all passed checks are identical
    checked_results = [r.get('match') for r in results.values() if r.get('match') is not None]
    all_match = all(r == True for r in checked_results)
    
    print("\n" + "=" * 80)
    if all_match:
        print("RESULT: Matrix tables are IDENTICAL (for all checks performed)")
    else:
        print("RESULT: Matrix tables DIFFER")
        print(f"\nOutput files written to: {os.getcwd()}")
    print("=" * 80)
    
    return results


#### Compare Dimensions

In [ ]:
# =============================================================================
# 1. COMPARE DIMENSIONS
# =============================================================================
dim_results = compare_dimensions(truth_annotated_mt, test_annotated_mt, "Original", "New")

# What happens:
# - Counts rows and columns in each MT
# - Prints comparison to screen
# - NO files are written
# 
# Returns:
# {
#     'match': True/False,
#     'n_rows_1': 10000,
#     'n_cols_1': 500,
#     'n_rows_2': 10000,
#     'n_cols_2': 500
# }

#### Compare Global Fields

In [ ]:
# =============================================================================
# 2. COMPARE GLOBAL FIELDS
# =============================================================================
global_results = compare_global_fields(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares all global-level fields (metadata that applies to whole MT)
# - Prints comparison to screen
# - IF differences found: writes "comparison_global_field_differences.txt"
#
# Returns:
# {
#     'match': True/False,
#     'n_diffs': 0 or number of differing fields,
#     'diff_fields': ['field1', 'field2', ...]  # list of fields that differ
# }

The fastest way to interpret the differences here is to feed it to AI (ensure that you do NOT copy any sample specific information and only unidentifiable metadata is included!)

The typical result is that the only difference is in the population order, for example:


* Original population order
  {'mid', 'afr', 'sas', 'eas', 'eur', 'amr'}

* New population order
  {'sas', 'afr', 'eas', 'eur', 'amr', 'mid'}

This difference will affect the descriptions for:
* pop_AN
* pop_AC_het
* pop_AC_hom
* pop_AF_hom
* pop_AF_het
* pop_hl_hist

The population order is non-deterministic. Capture the new orer here as you will need it later in the notebook. 

#### Compare Samples

In [ ]:
# =============================================================================
# 3. COMPARE SAMPLES (COLUMN KEYS)
# =============================================================================
sample_results = compare_samples(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares sample IDs (column keys)
# - Checks if same samples exist and if they're in same order
# - Prints comparison to screen
# - IF differences found: writes files listing unique samples
#   * "comparison_samples_only_in_Original.txt"
#   * "comparison_samples_only_in_New.txt"
#
# Returns:
# {
#     'match': True/False,           # same samples?
#     'order_match': True/False,     # same order?
#     'n_samples_1': 500,
#     'n_samples_2': 500,
#     'n_only_in_1': 0,
#     'n_only_in_2': 0
# }

#### Compare Column Fields

In [ ]:

# =============================================================================
# 4. COMPARE COLUMN FIELDS (SAMPLE METADATA)
# =============================================================================
col_results = compare_column_fields(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares metadata for each sample (age, contamination, coverage, etc.)
# - REQUIRES samples to match (skips if they don't)
# - Prints comparison to screen
# - IF differences found: writes "comparison_column_field_differences.tsv"
#   * Contains rows for samples with any differing metadata
#   * Shows both MT1 and MT2 values side-by-side
#
# Returns:
# {
#     'match': True/False/None,      # None if skipped
#     'n_fields_checked': 15,
#     'n_fields_diff': 2,
#     'diff_fields': {               # dict of field -> count of samples that differ
#         'contamination': 5,
#         'age': 2
#     }
# }
# OR if skipped:
# {
#     'match': None,
#     'skipped': True
# }

The following may need to be edited based on the output of the cloumn field comparison. These analyses address the differences we ahve seen before, but double check that there are no additional differences before running. 

In [ ]:
col_diff_df = pd.read_csv("/home/jupyter/Mitochondria_AoU_Workspace/edit/comparison_column_field_differences.tsv", sep="\t")

In [ ]:
col_diff_df.columns

In [ ]:
col_diff_df['over_85_mean'].sum()

In [ ]:
col_diff_df['mt2_over_85_mean'].sum()

In [ ]:
col_diff_df['bt_85_and_99_mean'].sum()

In [ ]:
col_diff_df['mt2_bt_85_and_99_mean'].sum()

In [ ]:
col_diff_df['contam_high_het'].sum()

In [ ]:
col_diff_df['mt2_contam_high_het'].sum()

In [ ]:
import matplotlib.pyplot as plt
df = col_diff_df

plt.figure(figsize=(8, 8))
plt.scatter(df['contam_high_het'], df['mt2_contam_high_het'], alpha=0.5)

# Add perfect match diagonal line
min_val = min(df['contam_high_het'].min(), df['mt2_contam_high_het'].min())
max_val = max(df['contam_high_het'].max(), df['mt2_contam_high_het'].max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect match')

plt.xlabel('MT1 contam_high_het')
plt.ylabel('MT2 contam_high_het')
plt.title('contam_high_het: MT1 vs MT2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Assuming your dataframe is called 'df'
# df = your_pandas_dataframe

# ============================================================================
# 1. IDENTIFY ROWS WITH DIFFERENCES
# ============================================================================

print("=" * 80)
print("ANALYZING COLUMN FIELD DIFFERENCES")
print("=" * 80)

# Create boolean masks for each difference
over_85_diff = col_diff_df['over_85_mean'] != col_diff_df['mt2_over_85_mean']
bt_85_99_diff = col_diff_df['bt_85_and_99_mean'] != col_diff_df['mt2_bt_85_and_99_mean']
contam_diff = col_diff_df['contam_high_het'] != col_diff_df['mt2_contam_high_het']

# Count differences
print(f"\nDifferences found:")
print(f"  over_85_mean: {over_85_diff.sum()} samples")
print(f"  bt_85_and_99_mean: {bt_85_99_diff.sum()} samples")
print(f"  contam_high_het: {contam_diff.sum()} samples")

# Find samples with ANY difference
any_diff = over_85_diff | bt_85_99_diff | contam_diff
print(f"\nTotal samples with any difference: {any_diff.sum()}")

# ============================================================================
# 2. FILTER TO ONLY ROWS WITH DIFFERENCES
# ============================================================================

df_diffs = col_diff_df[any_diff].copy()

# Add difference columns for easy viewing
df_diffs['over_85_mean_diff'] = df_diffs['over_85_mean'] - df_diffs['mt2_over_85_mean']
df_diffs['bt_85_and_99_mean_diff'] = df_diffs['bt_85_and_99_mean'] - df_diffs['mt2_bt_85_and_99_mean']
df_diffs['contam_high_het_diff'] = df_diffs['contam_high_het'] - df_diffs['mt2_contam_high_het']

print(f"\nFiltered to {len(df_diffs)} samples with differences")

# ============================================================================
# 3. SUMMARY STATISTICS FOR EACH FIELD
# ============================================================================

print("\n" + "=" * 80)
print("DIFFERENCE STATISTICS")
print("=" * 80)

for field in ['over_85_mean', 'bt_85_and_99_mean', 'contam_high_het']:
    diff_col = f'{field}_diff'
    
    # Only look at rows where this field actually differs
    field_diffs = df_diffs[df_diffs[field] != df_diffs[f'mt2_{field}']][diff_col]
    
    if len(field_diffs) > 0:
        print(f"\n{field}:")
        print(f"  Count: {len(field_diffs)}")
        print(f"  Min difference: {field_diffs.min():.6f}")
        print(f"  Max difference: {field_diffs.max():.6f}")
        print(f"  Mean difference: {field_diffs.mean():.6f}")
        print(f"  Median difference: {field_diffs.median():.6f}")
        print(f"  Std dev: {field_diffs.std():.6f}")
        print(f"  Abs mean difference: {field_diffs.abs().mean():.6f}")

# ============================================================================
# 4. VIEW EXAMPLES OF DIFFERENCES
# ============================================================================

print("\n" + "=" * 80)
print("EXAMPLE DIFFERENCES (first 10 samples)")
print("=" * 80)

# Select relevant columns for display
display_cols = [
    's', 
    'over_85_mean', 'mt2_over_85_mean', 'over_85_mean_diff',
    'bt_85_and_99_mean', 'mt2_bt_85_and_99_mean', 'bt_85_and_99_mean_diff',
    'contam_high_het', 'mt2_contam_high_het', 'contam_high_het_diff'
]

print(df_diffs[display_cols].head(10).to_string())

# ============================================================================
# 5. CHECK FOR PATTERNS
# ============================================================================

print("\n" + "=" * 80)
print("PATTERN ANALYSIS")
print("=" * 80)

# How many samples differ in multiple fields?
diff_counts = (
    over_85_diff[any_diff].astype(int) + 
    bt_85_99_diff[any_diff].astype(int) + 
    contam_diff[any_diff].astype(int)
)

print("\nSamples by number of differing fields:")
print(f"  1 field differs: {(diff_counts == 1).sum()}")
print(f"  2 fields differ: {(diff_counts == 2).sum()}")
print(f"  3 fields differ: {(diff_counts == 3).sum()}")

# Which combinations?
print("\nField combination patterns:")
print(f"  Only over_85_mean: {(over_85_diff & ~bt_85_99_diff & ~contam_diff).sum()}")
print(f"  Only bt_85_and_99_mean: {(~over_85_diff & bt_85_99_diff & ~contam_diff).sum()}")
print(f"  Only contam_high_het: {(~over_85_diff & ~bt_85_99_diff & contam_diff).sum()}")
print(f"  over_85 + bt_85_99: {(over_85_diff & bt_85_99_diff & ~contam_diff).sum()}")
print(f"  over_85 + contam: {(over_85_diff & ~bt_85_99_diff & contam_diff).sum()}")
print(f"  bt_85_99 + contam: {(~over_85_diff & bt_85_99_diff & contam_diff).sum()}")
print(f"  All three: {(over_85_diff & bt_85_99_diff & contam_diff).sum()}")

# ============================================================================
# 6. CHECK IF DIFFERENCES ARE CORRELATED
# ============================================================================

print("\n" + "=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

# Correlation between the difference magnitudes
diff_cols_only = df_diffs[['over_85_mean_diff', 'bt_85_and_99_mean_diff', 'contam_high_het_diff']]
correlation = diff_cols_only.corr()

print("\nCorrelation between difference magnitudes:")
print(correlation.to_string())

# ============================================================================
# 7. VISUALIZE DIFFERENCES (if matplotlib available)
# ============================================================================

try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Plot 1: over_85_mean
    ax = axes[0]
    field_data = df_diffs[df_diffs['over_85_mean'] != df_diffs['mt2_over_85_mean']]
    ax.scatter(field_data['over_85_mean'], field_data['mt2_over_85_mean'], alpha=0.5)
    ax.plot([field_data['over_85_mean'].min(), field_data['over_85_mean'].max()],
            [field_data['over_85_mean'].min(), field_data['over_85_mean'].max()],
            'r--', label='Perfect match')
    ax.set_xlabel('MT1 over_85_mean')
    ax.set_ylabel('MT2 over_85_mean')
    ax.set_title('over_85_mean Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: bt_85_and_99_mean
    ax = axes[1]
    field_data = df_diffs[df_diffs['bt_85_and_99_mean'] != df_diffs['mt2_bt_85_and_99_mean']]
    ax.scatter(field_data['bt_85_and_99_mean'], field_data['mt2_bt_85_and_99_mean'], alpha=0.5)
    ax.plot([field_data['bt_85_and_99_mean'].min(), field_data['bt_85_and_99_mean'].max()],
            [field_data['bt_85_and_99_mean'].min(), field_data['bt_85_and_99_mean'].max()],
            'r--', label='Perfect match')
    ax.set_xlabel('MT1 bt_85_and_99_mean')
    ax.set_ylabel('MT2 bt_85_and_99_mean')
    ax.set_title('bt_85_and_99_mean Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: contam_high_het
    ax = axes[2]
    field_data = df_diffs[df_diffs['contam_high_het'] != df_diffs['mt2_contam_high_het']]
    ax.scatter(field_data['contam_high_het'], field_data['mt2_contam_high_het'], alpha=0.5)
    ax.plot([field_data['contam_high_het'].min(), field_data['contam_high_het'].max()],
            [field_data['contam_high_het'].min(), field_data['contam_high_het'].max()],
            'r--', label='Perfect match')
    ax.set_xlabel('MT1 contam_high_het')
    ax.set_ylabel('MT2 contam_high_het')
    ax.set_title('contam_high_het Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Histogram of differences
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(df_diffs['over_85_mean_diff'].dropna(), bins=50, edgecolor='black')
    axes[0].set_xlabel('Difference')
    axes[0].set_ylabel('Count')
    axes[0].set_title('over_85_mean Differences')
    axes[0].axvline(0, color='red', linestyle='--', label='Zero difference')
    axes[0].legend()
    
    axes[1].hist(df_diffs['bt_85_and_99_mean_diff'].dropna(), bins=50, edgecolor='black')
    axes[1].set_xlabel('Difference')
    axes[1].set_ylabel('Count')
    axes[1].set_title('bt_85_and_99_mean Differences')
    axes[1].axvline(0, color='red', linestyle='--', label='Zero difference')
    axes[1].legend()
    
    axes[2].hist(df_diffs['contam_high_het_diff'].dropna(), bins=50, edgecolor='black')
    axes[2].set_xlabel('Difference')
    axes[2].set_ylabel('Count')
    axes[2].set_title('contam_high_het Differences')
    axes[2].axvline(0, color='red', linestyle='--', label='Zero difference')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("\nMatplotlib not available - skipping visualization")

# ============================================================================
# 8. SAVE FILTERED DATAFRAME FOR FURTHER ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("EXPORT OPTIONS")
print("=" * 80)

print(f"\nFiltered dataframe 'df_diffs' contains {len(df_diffs)} samples with differences")
print("You can:")
print("  - View it: df_diffs")
print("  - Export to CSV: df_diffs.to_csv('column_differences.csv', index=False)")
print("  - Look at specific sample: df_diffs[df_diffs['s'] == 'sample_id']")

#### Compare Variants 

In [ ]:
# =============================================================================
# 5. COMPARE VARIANTS (ROW KEYS)
# =============================================================================
variant_results = compare_variants(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares variant keys (locus, alleles)
# - Finds variants unique to each MT
# - Prints comparison to screen
# - IF differences found: writes TSV files
#   * "comparison_variants_only_in_Original.tsv"
#   * "comparison_variants_only_in_New.tsv"
#
# Returns:
# {
#     'match': True/False,
#     'n_variants_1': 10000,
#     'n_variants_2': 10000,
#     'n_only_in_1': 0,
#     'n_only_in_2': 0
# }

#### Compare Row Fields

##### Run the comparison and save the results

In [ ]:
# =============================================================================
# 6. COMPARE ROW FIELDS (VARIANT ANNOTATIONS)
# =============================================================================
row_results = compare_row_fields(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares annotations for each variant (filters, rsid, VEP, allele frequencies, etc.)
# - REQUIRES variants to match (skips if they don't)
# - Prints comparison to screen with progress indicator
# - IF differences found: writes "comparison_row_field_differences.tsv"
#   * Contains rows for variants with any differing annotations
#   * Shows both MT1 and MT2 values side-by-side
#
# Returns:
# {
#     'match': True/False/None,      # None if skipped
#     'n_fields_checked': 50,
#     'n_fields_diff': 3,
#     'diff_fields': {               # dict of field -> count of variants that differ
#         'rsid': 100,
#         'AN': 50,
#         'AF_hom': 25
#     }
# }
# OR if skipped:
# {
#     'match': None,
#     'skipped': True
# }

This is likely going to fail, however you will get a summary of differences. If your summary of differences matches the difference we have seen before, proceed through the notebook. If not, you may need to introduce new analyses to examine the new changes. 

Here's the summary of the changes we have seen before:

✗ 10 row fields have differences:                                               
  - pop_AN: 16,723 variants differ
  - pop_hl_hist: 10,921 variants differ
  - pop_AC_hom: 8,553 variants differ
  - pop_AF_hom: 8,553 variants differ
  - pop_AC_het: 6,264 variants differ
  - pop_AF_het: 6,264 variants differ
  - tlod_mean: 2,640 variants differ
  - mq_mean: 1,136 variants differ
  - hap_AF_hom: 113 variants differ
  - hap_AF_het: 113 variants differ

Now we're just going to compare the ones that have differences

In [ ]:
def compare_specific_row_fields(mt1, mt2, fields_to_check, mt1_name="MT1", mt2_name="MT2", 
                                output_prefix="comparison", save_intermediate=True):
    """
    Compare specific row fields, accounting for array order differences.
    
    Parameters:
    -----------
    mt1, mt2 : MatrixTable
    fields_to_check : list
        List of field names to compare
    mt1_name, mt2_name : str
        Names for output
    output_prefix : str
        Prefix for output files
    save_intermediate : bool
        If True, saves the differences table as a Hail Table
    
    Returns:
    --------
    dict with results AND the Hail Table of differences (if any)
    """
    print("\n" + "=" * 80)
    print(f"COMPARING {len(fields_to_check)} SPECIFIC ROW FIELDS")
    print("=" * 80)
    
    rows1_full = mt1.rows()
    rows2_full = mt2.rows()
    
    n_variants = mt1.count_rows()
    print(f"Comparing fields across {n_variants:,} variants...")
    print(f"Fields: {', '.join(fields_to_check)}")
    
    # Get population order from global fields
    pop_order_1 = None
    pop_order_2 = None
    reorder_indices = None
    
    if 'pop_order' in mt1.globals.dtype:
        pop_order_1 = list(mt1.globals.collect()[0].pop_order)
        pop_order_2 = list(mt2.globals.collect()[0].pop_order)
        print(f"\nPopulation orders:")
        print(f"  {mt1_name}: {pop_order_1}")
        print(f"  {mt2_name}: {pop_order_2}")
        
        if set(pop_order_1) == set(pop_order_2) and pop_order_1 != pop_order_2:
            print(f"  ⚠️  Same populations but different order - will reorder arrays")
            # Create mapping from MT2 order to MT1 order
            reorder_indices = [pop_order_2.index(pop) for pop in pop_order_1]
            print(f"  Reorder indices: {reorder_indices}")
    
    # Haplogroup order (if needed for hap_* fields)
    hap_order_1 = None
    hap_order_2 = None
    hap_reorder_indices = None
    
    if 'hap_order' in mt1.globals.dtype:
        hap_order_1 = list(mt1.globals.collect()[0].hap_order)
        hap_order_2 = list(mt2.globals.collect()[0].hap_order)
        
        if set(hap_order_1) == set(hap_order_2) and hap_order_1 != hap_order_2:
            print(f"\n  ⚠️  Haplogroup order also differs - will reorder hap arrays")
            hap_reorder_indices = [hap_order_2.index(hap) for hap in hap_order_1]
    
    # Population array fields that need reordering
    pop_array_fields = ['pop_AN', 'pop_AC_het', 'pop_AC_hom', 'pop_AF_hom', 'pop_AF_het', 'pop_hl_hist']
    
    # Haplogroup array fields that need reordering
    hap_array_fields = ['hap_AF_hom', 'hap_AF_het']
    
    # Join the row tables - only annotate the fields we care about
    print(f"\n  Joining row tables (annotating {len(fields_to_check)} fields)...")
    rows_joined = rows1_full.annotate(**{f"mt2_{field}": rows2_full[rows1_full.key][field] 
                                         for field in fields_to_check})
    
    # Reorder population arrays if needed
    if reorder_indices and any(f in pop_array_fields for f in fields_to_check):
        print(f"  Reordering population arrays...")
        for field in fields_to_check:
            if field in pop_array_fields:
                if field == 'pop_hl_hist':
                    # 2D array - reorder outer dimension
                    rows_joined = rows_joined.annotate(
                        **{f"mt2_{field}_reordered": hl.map(
                            lambda i: rows_joined[f"mt2_{field}"][reorder_indices[i]], 
                            hl.range(len(reorder_indices))
                        )}
                    )
                else:
                    # 1D array
                    rows_joined = rows_joined.annotate(
                        **{f"mt2_{field}_reordered": hl.map(
                            lambda i: rows_joined[f"mt2_{field}"][reorder_indices[i]], 
                            hl.range(len(reorder_indices))
                        )}
                    )
    
    # Reorder haplogroup arrays if needed
    if hap_reorder_indices and any(f in hap_array_fields for f in fields_to_check):
        print(f"  Reordering haplogroup arrays...")
        for field in fields_to_check:
            if field in hap_array_fields:
                rows_joined = rows_joined.annotate(
                    **{f"mt2_{field}_reordered": hl.map(
                        lambda i: rows_joined[f"mt2_{field}"][hap_reorder_indices[i]], 
                        hl.range(len(hap_reorder_indices))
                    )}
                )
    
    # Check for differences in each field
    print(f"\n  Checking each field for differences (after reordering where applicable)...")
    row_field_diffs = {}
    order_adjusted_fields = []
    
    for i, field in enumerate(fields_to_check, 1):
        print(f"    [{i}/{len(fields_to_check)}] Checking {field}...", end='\r')
        
        # Determine if we're using reordered version
        if field in pop_array_fields and reorder_indices:
            compare_field = f"mt2_{field}_reordered"
            order_adjusted_fields.append(field)
        elif field in hap_array_fields and hap_reorder_indices:
            compare_field = f"mt2_{field}_reordered"
            order_adjusted_fields.append(field)
        else:
            compare_field = f"mt2_{field}"
        
        # Count differences
        diff_count = rows_joined.aggregate(
            hl.agg.count_where(rows_joined[field] != rows_joined[compare_field])
        )
        
        if diff_count > 0:
            row_field_diffs[field] = diff_count
    
    print(" " * 80, end='\r')  # Clear progress line
    
    # Report which fields were order-adjusted
    if order_adjusted_fields:
        print(f"\n  ℹ️  Order-adjusted fields: {', '.join(order_adjusted_fields)}")
    
    # Report results
    row_fields_match = len(row_field_diffs) == 0
    
    results = {
        'match': row_fields_match,
        'n_fields_checked': len(fields_to_check),
        'n_fields_diff': len(row_field_diffs),
        'diff_fields': row_field_diffs,
        'order_adjusted_fields': order_adjusted_fields,
        'diff_table': None
    }
    
    print(f"\n  Results after order adjustment:")
    if row_fields_match:
        print("  ✓ All fields match (differences were due to ordering)")
    else:
        print(f"  ✗ {len(row_field_diffs)} fields still have differences:")
        for field, count in sorted(row_field_diffs.items(), key=lambda x: x[1], reverse=True):
            marker = " (order-adjusted)" if field in order_adjusted_fields else " (NOT order-related)"
            print(f"    - {field}: {count:,} variants differ{marker}")
    
    # If there are still differences, create and save the diff table
    if not row_fields_match:
        print(f"\n  Creating table of remaining differences...")
        
        # Build filter condition
        filter_conditions = []
        for field in row_field_diffs.keys():
            if field in order_adjusted_fields:
                if field in pop_array_fields and reorder_indices:
                    compare_field = f"mt2_{field}_reordered"
                elif field in hap_array_fields and hap_reorder_indices:
                    compare_field = f"mt2_{field}_reordered"
                else:
                    compare_field = f"mt2_{field}"
            else:
                compare_field = f"mt2_{field}"
            
            filter_conditions.append(rows_joined[field] != rows_joined[compare_field])
        
        rows_diff = rows_joined.filter(hl.any(filter_conditions))
        
        # Count the diff table
        n_diff_rows = rows_diff.count()
        print(f"  Diff table has {n_diff_rows:,} rows")
        
        
        # Store table in results
        results['diff_table'] = rows_diff
        
    return results


# Usage
fields_with_diffs = [
    'pop_AN',
    'pop_hl_hist',
    'tlod_mean',
    'pop_AF_hom',
    'pop_AC_hom',
    'mq_mean',
    'pop_AF_het',
    'pop_AC_het',
    'hap_AF_hom',
    'hap_AF_het'
]


row_results = compare_specific_row_fields(
    truth_annotated_mt, 
    test_annotated_mt,
    fields_with_diffs,
    "Original",
    "New",
    "comparison",
    save_intermediate=True
)

# Check which fields still differ after order adjustment
if row_results['n_fields_diff'] > 0:
    print("\n" + "=" * 80)
    print("FIELDS WITH REAL DIFFERENCES (not due to ordering):")
    print("=" * 80)
    for field, count in row_results['diff_fields'].items():
        if field not in row_results['order_adjusted_fields']:
            print(f"  {field}: {count:,} variants")

Although the code above claims to account for differences in the population order, it does not. The issue is that the sets are read in as python lists and that process reorders them to match. We therefore need to more carefully consider the order of the populations and the role it is playing in the observed differences.  

##### Subset to just the fields with differences

In [ ]:
diff_table = row_results['diff_table']

# Use key_by() to drop key status first, then select
diff_table_subset = diff_table.key_by().select(
    # Row keys
    'locus',
    'alleles',
    # MT1 values
    'pop_AN',
    'pop_hl_hist',
    'tlod_mean',
    'pop_AF_hom',
    'pop_AC_hom',
    'mq_mean',
    'pop_AF_het',
    'pop_AC_het',
    'hap_AF_hom',
    'hap_AF_het',
    # MT2 values
    'mt2_pop_AN',
    'mt2_pop_hl_hist',
    'mt2_tlod_mean',
    'mt2_pop_AF_hom',
    'mt2_pop_AC_hom',
    'mt2_mq_mean',
    'mt2_pop_AF_het',
    'mt2_pop_AC_het',
    'mt2_hap_AF_hom',
    'mt2_hap_AF_het',
)

print(f"Subset table has {diff_table_subset.count()} rows")

In [ ]:
diff_table_subset.show()

##### Convert the diff table to pandas

In [ ]:
# Convert to pandas
df_row_diffs = diff_table_subset.to_pandas()
print(f"Shape: {df_row_diffs.shape}")
df_row_diffs.head()

In [ ]:
list(df_row_diffs.columns)

##### Let's see if we can explain some of these differences with the population order

Update the following population orders based on your findings from earlier

In [ ]:
# Original population order {'mid', 'afr', 'sas', 'eas', 'eur', 'amr'}

# New population order {'sas', 'afr', 'eas', 'eur', 'amr', 'mid'}

In [ ]:
import numpy as np

# ============================================================================
# 1. DEFINE POPULATION ORDERS AND REORDER INDICES
# ============================================================================

mt1_pop_order = ['mid', 'afr', 'sas', 'eas', 'eur', 'amr'] #UPDATE HERE#
mt2_pop_order = ['sas', 'afr', 'eas', 'eur', 'amr', 'mid'] #UPDATE HERE#

# For each position in mt1 order, find where that population is in mt2
reorder_indices = [mt2_pop_order.index(pop) for pop in mt1_pop_order]

print("Population reordering:")
for i, (pop, idx) in enumerate(zip(mt1_pop_order, reorder_indices)):
    print(f"  mt1 position {i} ({pop}) <- mt2 position {idx} ({mt2_pop_order[idx]})")

# ============================================================================
# 2. REORDER MT2 POPULATION ARRAYS
# ============================================================================

pop_fields = ['pop_AN', 'pop_AF_hom', 'pop_AC_hom', 'pop_AF_het', 'pop_AC_het', 'pop_hl_hist']

print("\nReordering mt2 population arrays...")

for field in pop_fields:
    mt2_col = f'mt2_{field}'
    mt2_col_reordered = f'mt2_{field}_reordered'
    
    if mt2_col in df_row_diffs.columns:
        df_row_diffs[mt2_col_reordered] = df_row_diffs[mt2_col].apply(
            lambda x: [x[i] for i in reorder_indices] if isinstance(x, list) else x
        )
        print(f"  ✓ {mt2_col_reordered} created")
    else:
        print(f"  ✗ {mt2_col} not found in dataframe")

# ============================================================================
# 3. VERIFY REORDERING ON A SINGLE EXAMPLE
# ============================================================================

print("\nVerification on first row:")
row = df_row_diffs.iloc[0]
print(f"  mt1 pop_AN:           {row['pop_AN']}")
print(f"  mt2 pop_AN (original):  {row['mt2_pop_AN']}")
print(f"  mt2 pop_AN (reordered): {row['mt2_pop_AN_reordered']}")
print(f"\n  mt1 order: {dict(zip(mt1_pop_order, row['pop_AN']))}")
print(f"  mt2 order (reordered): {dict(zip(mt1_pop_order, row['mt2_pop_AN_reordered']))}")

# ============================================================================
# 4. COMPARE MT1 VS REORDERED MT2
# ============================================================================

print("\n" + "=" * 80)
print("COMPARISON AFTER REORDERING")
print("=" * 80)

def arrays_equal(a, b):
    """Compare two arrays element-wise, handling None/NaN."""
    if a is None or b is None:
        return a == b
    if not isinstance(a, list) or not isinstance(b, list):
        return a == b
    if len(a) != len(b):
        return False
    return all(
        (x == y) or (x is None and y is None) or 
        (isinstance(x, float) and isinstance(y, float) and np.isnan(x) and np.isnan(y))
        for x, y in zip(a, b)
    )

summary = {}

for field in pop_fields:
    mt2_col_reordered = f'mt2_{field}_reordered'
    
    if mt2_col_reordered not in df_row_diffs.columns:
        continue
    
    # Compare element-wise
    matches = df_row_diffs.apply(
        lambda row: arrays_equal(row[field], row[mt2_col_reordered]), axis=1
    )
    
    n_diff = (~matches).sum()
    n_total = len(df_row_diffs)
    summary[field] = n_diff
    
    if n_diff == 0:
        print(f"✓ {field}: MATCH after reordering")
    else:
        print(f"✗ {field}: {n_diff}/{n_total} variants still differ after reordering")

# ============================================================================
# 5. INVESTIGATE ANY REMAINING DIFFERENCES
# ============================================================================

fields_still_diff = {f: c for f, c in summary.items() if c > 0}

if not fields_still_diff:
    print("\n✓ ALL population fields match after reordering!")
    print("  The differences were entirely due to population order.")
else:
    print(f"\n{len(fields_still_diff)} fields still have differences after reordering:")
    
    for field in fields_still_diff:
        mt2_col_reordered = f'mt2_{field}_reordered'
        
        matches = df_row_diffs.apply(
            lambda row: arrays_equal(row[field], row[mt2_col_reordered]), axis=1
        )
        still_diff = df_row_diffs[~matches]
        
        print(f"\n  {field} - {len(still_diff)} variants:")
        
        # Show a few examples
        for _, row in still_diff.head(3).iterrows():
            print(f"    Locus: {row['locus']}, Alleles: {row['alleles']}")
            print(f"      mt1:            {row[field]}")
            print(f"      mt2 (reordered): {row[mt2_col_reordered]}")
            
            # Show per-population differences
            if isinstance(row[field], list) and isinstance(row[mt2_col_reordered], list):
                diffs = [(mt1_pop_order[i], row[field][i], row[mt2_col_reordered][i])
                         for i in range(len(mt1_pop_order))
                         if row[field][i] != row[mt2_col_reordered][i]]
                if diffs:
                    print(f"      Per-pop differences (pop, mt1, mt2):")
                    for pop, v1, v2 in diffs:
                        print(f"        {pop}: {v1} vs {v2}")

# ============================================================================
# 6. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"{'Field':<20} {'Diffs before':>15} {'Diffs after':>15} {'Resolved?':>12}")
print("-" * 65)

original_counts = {
    'pop_AN': 3356, 'pop_hl_hist': 2325, 'pop_AF_hom': 2122,
    'pop_AC_hom': 2119, 'pop_AF_het': 379, 'pop_AC_het': 378
}

for field in pop_fields:
    orig = original_counts.get(field, 'N/A')
    after = summary.get(field, 'N/A')
    resolved = "✓ YES" if after == 0 else "✗ NO"
    print(f"{field:<20} {str(orig):>15} {str(after):>15} {resolved:>12}")

##### Ok, so now what are the remaining differences that we need to address

Remaining differences
  * tlod_mean: 2,236 variants
  * mq_mean: 383 variants
  * hap_AF_hom: 32 variants
  * hap_AF_het: 32 variants

In [ ]:
cols_of_interest = [
    'locus', 
    'alleles',
    'hap_AF_hom',
    'hap_AF_het',
    'mt2_hap_AF_hom',
    'mt2_hap_AF_het'
]
hap_diff_rows = df_row_diffs[
    (df_row_diffs['hap_AF_hom'] != df_row_diffs['mt2_hap_AF_hom']) |
    (df_row_diffs['hap_AF_het'] != df_row_diffs['mt2_hap_AF_het'])
][cols_of_interest]

In [ ]:
mt1_hap_order = [
    'A','B','C','D','F','G','H','HV','I','J','K',
    'L0','L1','L2','L3','L4','M','N','R','T','U','V','W','X','Y'
]
mt2_hap_order = [
    'A','B','C','D','F','G','H','HV','I','J','K',
    'L0','L1','L2','L3','L4','M','N','R','T','U','V','W','X','Y'
]

In [ ]:
# Get a variant that differs in hap_AF_hom
row = hap_diff_rows.iloc[0]

mt1_vals = list(row['hap_AF_hom'])
mt2_vals = list(row['mt2_hap_AF_hom'])

print(f"Variant: {row['locus']}")
print(f"\nFull side-by-side comparison:")
print(f"{'Pos':<5} {'Haplogroup':<12} {'MT1':>12} {'MT2':>12} {'Match':>8}")
print("-" * 52)
for i, hap in enumerate(mt1_hap_order):
    v1 = mt1_vals[i]
    v2 = mt2_vals[i]
    match = "✓" if v1 == v2 else "✗"
    print(f"{i:<5} {hap:<12} {str(v1):>12} {str(v2):>12} {match:>8}")

# Key question: are the MT2 values just a reordering of MT1 values?
print(f"\nAre the same values present in both (just possibly reordered)?")
print(f"  MT1 sorted: {sorted([v for v in mt1_vals if v is not None])}")
print(f"  MT2 sorted: {sorted([v for v in mt2_vals if v is not None])}")
same_values = sorted([v for v in mt1_vals if v is not None]) == \
              sorted([v for v in mt2_vals if v is not None])
print(f"  Same values: {same_values}")

# If same_values is True, it's an ordering issue
# If same_values is False, it's a real data difference
if same_values:
    print("\n⚠️  Same values present - this is likely an ordering issue!")
    print("    Need to find the correct haplogroup order mapping")
else:
    print("\n✓ Different values present - this is a REAL data difference")

In [ ]:
import numpy as np

def arrays_equal_nan_safe(a, b):
    """Compare two arrays element-wise, treating nan == nan as True."""
    if a is None or b is None:
        return a == b
    if not isinstance(a, list) or not isinstance(b, list):
        return a == b
    if len(a) != len(b):
        return False
    for x, y in zip(a, b):
        # Both nan -> treat as equal
        if isinstance(x, float) and isinstance(y, float):
            if np.isnan(x) and np.isnan(y):
                continue
        # Otherwise normal comparison
        if x != y:
            return False
    return True

# Re-check hap fields with nan-safe comparison
print("HAP FIELD COMPARISON (nan-safe)")
print("=" * 80)

for field in ['hap_AF_hom', 'hap_AF_het']:
    matches = df_row_diffs.apply(
        lambda row: arrays_equal_nan_safe(row[field], row[f'mt2_{field}']), axis=1
    )
    n_diff = (~matches).sum()
    
    if n_diff == 0:
        print(f"✓ {field}: MATCH (differences were due to nan comparison)")
    else:
        print(f"✗ {field}: {n_diff} variants still differ after nan-safe comparison")
        still_diff = df_row_diffs[~matches]
        print(f"\n  Examples:")
        for _, row in still_diff.head(3).iterrows():
            print(f"  Locus: {row['locus']}")
            print(f"    MT1: {row[field]}")
            print(f"    MT2: {row[f'mt2_{field}']}")

##### OK, so now we're down to just 2 mean differences

Remaining differences

* tlod_mean: 2,236 variants
* mq_mean: 383 variants

In [ ]:
df_row_diffs['tlod_mean'].sum()

In [ ]:
df_row_diffs['mt2_tlod_mean'].sum()

In [ ]:
df_row_diffs['mq_mean'].sum()

In [ ]:
df_row_diffs['mt2_mq_mean'].sum()

#### Compare Entry Fields

In [ ]:
# =============================================================================
# 7. COMPARE ENTRY FIELDS (GENOTYPE DATA)
# =============================================================================
entry_results = compare_entry_fields(truth_annotated_mt, test_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares genotype-level data (DP, AD, HL, TLOD, etc.)
# - REQUIRES both samples AND variants to match (skips otherwise)
# - Prints comparison to screen with progress indicator
# - IF differences found: writes "comparison_entry_differences.tsv"
#   * Contains entries (variant x sample combinations) with differences
#   * Shows both MT1 and MT2 values side-by-side
#   * WARNING: This file can be LARGE if many entries differ
#
# Returns:
# {
#     'match': True/False/None,      # None if skipped
#     'n_fields_checked': 15,
#     'n_fields_diff': 2,
#     'diff_fields': {               # dict of field -> count of entries that differ
#         'DP': 1000,
#         'HL': 500
#     }
# }
# OR if skipped:
# {
#     'match': None,
#     'skipped': True
# }

# Compare filtered annotated VCF matrices

### Original analysis

#### Compare Counts 1000

In [ ]:
truth_filt_annotated_mt = truth_filt_annotated_mt_50k
test_filt_annotated_mt = test_filt_annotated_mt_50k

In [ ]:
# Compare counts - note this is computationally expensive
print("Counts for MT1:")
print(truth_filt_annotated_mt.count())

print("Counts for MT2:")
print(test_filt_annotated_mt.count())

#### Compare Schemas 1000

In [ ]:
# Examine the first table
print("--- Describing MT1 ---")
og_filt_annotated_mt1.describe()

# Examine the second table
print("--- Describing MT2 ---")
new_filt_annotated_mt1.describe()

In [ ]:
def compare_schemas(mt1, mt2, mt1_name="MT1", mt2_name="MT2"):
    """
    Systematically compare schemas of two Hail matrix tables.
    
    Parameters:
    -----------
    mt1, mt2 : MatrixTable
        The two matrix tables to compare
    mt1_name, mt2_name : str
        Names for the matrix tables in output
    
    Returns:
    --------
    dict : Comparison results
    """
    
    results = {
        'global_fields_match': True,
        'column_fields_match': True,
        'row_fields_match': True,
        'entry_fields_match': True,
        'keys_match': True,
        'differences': []
    }
    
    print("=" * 80)
    print(f"SCHEMA COMPARISON: {mt1_name} vs {mt2_name}")
    print("=" * 80)
    
    # Helper function to get field types as dict
    def get_field_dict(fields):
        return {name: str(dtype) for name, dtype in fields.items()}
    
    # 1. Compare Global Fields
    print("\n1. GLOBAL FIELDS:")
    print("-" * 80)
    
    global1 = get_field_dict(mt1.globals.dtype)
    global2 = get_field_dict(mt2.globals.dtype)
    
    if global1 == global2:
        print(f"✓ Global fields match ({len(global1)} fields)")
    else:
        print(f"✗ Global fields differ")
        results['global_fields_match'] = False
        
        only_in_mt1 = set(global1.keys()) - set(global2.keys())
        only_in_mt2 = set(global2.keys()) - set(global1.keys())
        common = set(global1.keys()) & set(global2.keys())
        type_diffs = [f for f in common if global1[f] != global2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {global1[field]}")
                results['differences'].append(f"Global field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {global2[field]}")
                results['differences'].append(f"Global field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {global1[field]}")
                print(f"      {mt2_name}: {global2[field]}")
                results['differences'].append(f"Global field '{field}' has different types")
    
    # 2. Compare Column Fields
    print("\n2. COLUMN FIELDS:")
    print("-" * 80)
    
    col1 = get_field_dict(mt1.col.dtype)
    col2 = get_field_dict(mt2.col.dtype)
    
    if col1 == col2:
        print(f"✓ Column fields match ({len(col1)} fields)")
    else:
        print(f"✗ Column fields differ")
        results['column_fields_match'] = False
        
        only_in_mt1 = set(col1.keys()) - set(col2.keys())
        only_in_mt2 = set(col2.keys()) - set(col1.keys())
        common = set(col1.keys()) & set(col2.keys())
        type_diffs = [f for f in common if col1[f] != col2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {col1[field]}")
                results['differences'].append(f"Column field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {col2[field]}")
                results['differences'].append(f"Column field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {col1[field]}")
                print(f"      {mt2_name}: {col2[field]}")
                results['differences'].append(f"Column field '{field}' has different types")
    
    # 3. Compare Row Fields
    print("\n3. ROW FIELDS:")
    print("-" * 80)
    
    row1 = get_field_dict(mt1.row.dtype)
    row2 = get_field_dict(mt2.row.dtype)
    
    if row1 == row2:
        print(f"✓ Row fields match ({len(row1)} fields)")
    else:
        print(f"✗ Row fields differ")
        results['row_fields_match'] = False
        
        only_in_mt1 = set(row1.keys()) - set(row2.keys())
        only_in_mt2 = set(row2.keys()) - set(row1.keys())
        common = set(row1.keys()) & set(row2.keys())
        type_diffs = [f for f in common if row1[f] != row2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {row1[field]}")
                results['differences'].append(f"Row field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {row2[field]}")
                results['differences'].append(f"Row field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {row1[field]}")
                print(f"      {mt2_name}: {row2[field]}")
                results['differences'].append(f"Row field '{field}' has different types")
    
    # 4. Compare Entry Fields
    print("\n4. ENTRY FIELDS:")
    print("-" * 80)
    
    entry1 = get_field_dict(mt1.entry.dtype)
    entry2 = get_field_dict(mt2.entry.dtype)
    
    if entry1 == entry2:
        print(f"✓ Entry fields match ({len(entry1)} fields)")
    else:
        print(f"✗ Entry fields differ")
        results['entry_fields_match'] = False
        
        only_in_mt1 = set(entry1.keys()) - set(entry2.keys())
        only_in_mt2 = set(entry2.keys()) - set(entry1.keys())
        common = set(entry1.keys()) & set(entry2.keys())
        type_diffs = [f for f in common if entry1[f] != entry2[f]]
        
        if only_in_mt1:
            print(f"\n  Fields only in {mt1_name}: {len(only_in_mt1)}")
            for field in sorted(only_in_mt1):
                print(f"    {field}: {entry1[field]}")
                results['differences'].append(f"Entry field '{field}' only in {mt1_name}")
        
        if only_in_mt2:
            print(f"\n  Fields only in {mt2_name}: {len(only_in_mt2)}")
            for field in sorted(only_in_mt2):
                print(f"    {field}: {entry2[field]}")
                results['differences'].append(f"Entry field '{field}' only in {mt2_name}")
        
        if type_diffs:
            print(f"\n  Fields with different types: {len(type_diffs)}")
            for field in sorted(type_diffs):
                print(f"    {field}:")
                print(f"      {mt1_name}: {entry1[field]}")
                print(f"      {mt2_name}: {entry2[field]}")
                results['differences'].append(f"Entry field '{field}' has different types")
    
    # 5. Compare Keys
    print("\n5. KEY FIELDS:")
    print("-" * 80)
    
    col_key1 = list(mt1.col_key.dtype)
    col_key2 = list(mt2.col_key.dtype)
    row_key1 = list(mt1.row_key.dtype)
    row_key2 = list(mt2.row_key.dtype)
    
    keys_match = True
    
    if col_key1 == col_key2:
        print(f"✓ Column keys match: {list(mt1.col_key.dtype.keys())}")
    else:
        print(f"✗ Column keys differ")
        print(f"  {mt1_name}: {list(mt1.col_key.dtype.keys())}")
        print(f"  {mt2_name}: {list(mt2.col_key.dtype.keys())}")
        results['differences'].append("Column keys differ")
        keys_match = False
    
    if row_key1 == row_key2:
        print(f"✓ Row keys match: {list(mt1.row_key.dtype.keys())}")
    else:
        print(f"✗ Row keys differ")
        print(f"  {mt1_name}: {list(mt1.row_key.dtype.keys())}")
        print(f"  {mt2_name}: {list(mt2.row_key.dtype.keys())}")
        results['differences'].append("Row keys differ")
        keys_match = False
    
    results['keys_match'] = keys_match
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY:")
    print("=" * 80)
    
    all_match = (results['global_fields_match'] and 
                 results['column_fields_match'] and 
                 results['row_fields_match'] and 
                 results['entry_fields_match'] and 
                 results['keys_match'])
    
    if all_match:
        print("✓ Schemas are IDENTICAL")
    else:
        print("✗ Schemas DIFFER")
        print(f"\nTotal differences found: {len(results['differences'])}")
        
        if not results['global_fields_match']:
            print("  - Global fields differ")
        if not results['column_fields_match']:
            print("  - Column fields differ")
        if not results['row_fields_match']:
            print("  - Row fields differ")
        if not results['entry_fields_match']:
            print("  - Entry fields differ")
        if not results['keys_match']:
            print("  - Keys differ")
    
    return results


# Example usage:
if __name__ == "__main__":
    
    # Compare schemas
    schema_results = compare_schemas(truth_filt_annotated_mt, test_filt_annotated_mt, "Original MT", "New MT")
    
    # Access results
    print(f"\nSchemas identical? {all(schema_results[k] for k in ['global_fields_match', 'column_fields_match', 'row_fields_match', 'entry_fields_match', 'keys_match'])}")

In [ ]:
# =============================================================================
# 3. COMPARE SAMPLES (COLUMN KEYS)
# =============================================================================
sample_results = compare_samples(truth_filt_annotated_mt, test_filt_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares sample IDs (column keys)
# - Checks if same samples exist and if they're in same order
# - Prints comparison to screen
# - IF differences found: writes files listing unique samples
#   * "comparison_samples_only_in_Original.txt"
#   * "comparison_samples_only_in_New.txt"
#
# Returns:
# {
#     'match': True/False,           # same samples?
#     'order_match': True/False,     # same order?
#     'n_samples_1': 500,
#     'n_samples_2': 500,
#     'n_only_in_1': 0,
#     'n_only_in_2': 0
# }

In [ ]:
# =============================================================================
# 5. COMPARE VARIANTS (ROW KEYS)
# =============================================================================
variant_results = compare_variants(truth_filt_annotated_mt, test_filt_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares variant keys (locus, alleles)
# - Finds variants unique to each MT
# - Prints comparison to screen
# - IF differences found: writes TSV files
#   * "comparison_variants_only_in_Original.tsv"
#   * "comparison_variants_only_in_New.tsv"
#
# Returns:
# {
#     'match': True/False,
#     'n_variants_1': 10000,
#     'n_variants_2': 10000,
#     'n_only_in_1': 0,
#     'n_only_in_2': 0
# }

In [ ]:
# =============================================================================
# 7. COMPARE ENTRY FIELDS (GENOTYPE DATA)
# =============================================================================
entry_results = compare_entry_fields(truth_filt_annotated_mt, test_filt_annotated_mt, "Original", "New", "comparison")

# What happens:
# - Compares genotype-level data (DP, AD, HL, TLOD, etc.)
# - REQUIRES both samples AND variants to match (skips otherwise)
# - Prints comparison to screen with progress indicator
# - IF differences found: writes "comparison_entry_differences.tsv"
#   * Contains entries (variant x sample combinations) with differences
#   * Shows both MT1 and MT2 values side-by-side
#   * WARNING: This file can be LARGE if many entries differ
#
# Returns:
# {
#     'match': True/False/None,      # None if skipped
#     'n_fields_checked': 15,
#     'n_fields_diff': 2,
#     'diff_fields': {               # dict of field -> count of entries that differ
#         'DP': 1000,
#         'HL': 500
#     }
# }
# OR if skipped:
# {
#     'match': None,
#     'skipped': True
# }